In [4]:
import pandas as pd

# Path to the source Excel file
file_path = "D:\WORK - MAIN\PROJECTS\Smart Traffic Data Engineering & Analytics Pipeline\smart_traffic_dataset-v2.xlsx"

# Load both sheets
traffic_df = pd.read_excel(file_path, sheet_name="Traffic_Logs_Raw")
location_df = pd.read_excel(file_path, sheet_name="Location_Lookup")

# Check that the data was loaded correctly
print("Traffic data:")
print(traffic_df.head())

print("\nLocation lookup:")
print(location_df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\WORK - MAIN\\PROJECTS\\Smart Traffic Data Engineering & Analytics Pipeline\\smart_traffic_dataset-v2.xlsx'

In [ ]:
traffic_clean = traffic_df.copy()

In [ ]:
traffic_clean["Vehicle_Type"] = traffic_clean["Vehicle_Type"].str.strip()
traffic_clean["Weather_Condition"] = traffic_clean["Weather_Condition"].str.strip()

In [ ]:
traffic_clean["Vehicle_Type"] = traffic_clean["Vehicle_Type"].str.lower()
traffic_clean["Weather_Condition"] = traffic_clean["Weather_Condition"].str.lower()

In [ ]:
traffic_clean["Timestamp"] = pd.to_datetime(traffic_clean["Timestamp"], errors="coerce")

In [ ]:
traffic_clean = traffic_clean.dropna(subset=["Timestamp"])

In [ ]:
traffic_clean = traffic_clean.dropna(subset=["Location_ID"])

In [ ]:
traffic_clean["Location_ID"] = traffic_clean["Location_ID"].astype("string").str.strip().str.upper()
location_df["Location_ID"] = location_df["Location_ID"].astype("string").str.strip().str.upper()

In [ ]:
invalid_location_ids = traffic_clean[
    ~traffic_clean["Location_ID"].isin(location_df["Location_ID"])
]

print("\nUnmatched Location_ID records after standardization:")
print(len(invalid_location_ids))


Unmatched Location_ID records after standardization:
0


In [ ]:
traffic_clean = traffic_clean[
    traffic_clean["Location_ID"].isin(location_df["Location_ID"])
]

In [ ]:
traffic_clean["Speed_kmh"] = traffic_clean["Speed_kmh"].fillna(
    traffic_clean["Speed_kmh"].median()
)

In [ ]:
print("Minimum Speed:", traffic_clean["Speed_kmh"].min())
print("Maximum Speed:", traffic_clean["Speed_kmh"].max())

print("\nRows with Speed <= 0:")
print(
    traffic_clean[
        traffic_clean["Speed_kmh"] <= 0
    ][["Record_ID", "Vehicle_Type", "Location_ID", "Speed_kmh"]]
)

Minimum Speed: 5.0
Maximum Speed: 118.3

Rows with Speed <= 0:
Empty DataFrame
Columns: [Record_ID, Vehicle_Type, Location_ID, Speed_kmh]
Index: []


In [ ]:
traffic_clean["Traffic_Volume"] = traffic_clean["Traffic_Volume"].fillna(
    traffic_clean["Traffic_Volume"].median()
)

In [ ]:
traffic_clean["Weather_Condition"] = traffic_clean["Weather_Condition"].fillna(
    traffic_clean["Weather_Condition"].mode()[0]
)

In [ ]:
print("Duplicate rows:")
print(traffic_clean.duplicated().sum())

print("\nDuplicate Record_IDs:")
print(traffic_clean["Record_ID"].duplicated().sum())

Duplicate rows:
0

Duplicate Record_IDs:
0


In [ ]:
print("Negative Speed values:")
print((traffic_clean["Speed_kmh"] < 0).sum())

print("\nNegative Traffic Volume values:")
print((traffic_clean["Traffic_Volume"] < 0).sum())

Negative Speed values:
0

Negative Traffic Volume values:
0


In [ ]:
print("Original rows:", len(traffic_df))
print("Cleaned rows:", len(traffic_clean))
print("Rows removed:", len(traffic_df) - len(traffic_clean))

Original rows: 12000
Cleaned rows: 12000
Rows removed: 0


In [ ]:
print("Missing values in Location Lookup:")
print(location_df.isna().sum())

print("\nDuplicate Location IDs:")
print(location_df["Location_ID"].duplicated().sum())

Missing values in Location Lookup:
Location_ID        0
Zone_Name          0
Road_Type          0
Speed_Limit_kmh    0
dtype: int64

Duplicate Location IDs:
0


In [ ]:
traffic_enriched = traffic_clean.merge(
    location_df,
    on="Location_ID",
    how="left"
)

In [ ]:
print("Missing Zone_Name values:")
print(traffic_enriched["Zone_Name"].isna().sum())

Missing Zone_Name values:
0


In [ ]:
print("Negative Speed values:")
print((traffic_enriched["Speed_kmh"] < 0).sum())

print("\nNegative Traffic Volume values:")
print((traffic_enriched["Traffic_Volume"] < 0).sum())

print("\nNegative Speed Limit values:")
print((traffic_enriched["Speed_Limit_kmh"] < 0).sum())

Negative Speed values:
0

Negative Traffic Volume values:
0

Negative Speed Limit values:
0


In [ ]:
final_traffic = traffic_enriched.copy()

In [ ]:
print("Missing values in final dataset:")
print(final_traffic.isna().sum())

Missing values in final dataset:


NameError: name 'final_traffic' is not defined

In [ ]:
final_traffic.to_csv("final_traffic_data.csv", index=False)

In [ ]:
location_df.to_csv("locations.csv", index=False)

In [ ]:
traffic_to_load = traffic_clean.copy()

In [ ]:
traffic_to_load.to_csv("traffic_logs.csv", index=False)

In [ ]:
ml_data = final_traffic.copy()

ml_data.head()

NameError: name 'final_traffic' is not defined

In [ ]:
ml_data["Traffic_Hour"] = ml_data["Timestamp"].dt.hour

ml_data[[
    "Timestamp",
    "Traffic_Hour",
    "Traffic_Volume"
]].head()

In [ ]:
features = [
    "Speed_kmh",
    "Traffic_Hour",
    "Vehicle_Type",
    "Weather_Condition",
    "Location_ID",
    "Road_Type",
    "Speed_Limit_kmh"
]

X = ml_data[features]

y = ml_data["Traffic_Volume"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "Vehicle_Type",
    "Weather_Condition",
    "Location_ID",
    "Road_Type"
]

numeric_features = [
    "Speed_kmh",
    "Traffic_Hour",
    "Speed_Limit_kmh"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

y_pred[:10]

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R² Score:", round(r2, 4))